# pGrAdd with BensonGAF Library

This notebook demonstrates how to use the **BensonGAF** group-additivity library
(developed as part of *Computational Methods for Rapid Fluorocarbon Characterization*)
with the [pGrAdd](https://github.com/VlachosGroup/PythonGroupAdditivity) package.

It covers two workflows:
1. **Single-molecule estimation** — compute H, S, and Cp for one SMILES string.
2. **Batch validation** — compare pGrAdd predictions against DFT-derived thermochemical data
   for a set of fluorocarbon molecules and report absolute errors.

### Prerequisites
Install all required packages (see `requirements.txt`) and copy the `BensonGAF/` folder
into your pGrAdd data directory before running this notebook.  
See the repository README for step-by-step setup instructions.

## 1. Single-Molecule Thermochemistry Estimation

Load the `BensonGAF` library and estimate thermochemical properties for a single
molecule specified as a SMILES string.

* `GetDescriptors(smiles)` — identifies and counts Benson groups in the molecule.
* `Estimate(descriptors, 'thermochem')` — sums group contributions to give H, S, Cp.
* Results are unit-convertible: `get_H(units='kJ/mol', T=300)`, etc.

In [24]:
from pgradd.GroupAdd.Library import GroupLibrary
import pgradd.ThermoChem
lib = GroupLibrary.Load('BensonGAF')
descriptors = lib.GetDescriptors('C(F)(F)(=C(F)(F))')
print(descriptors)
thermochem = lib.Estimate(descriptors,'thermochem')
print(thermochem.get_H(units='kJ/mol',T=300))
print(thermochem.get_S(units='cal/mol/K',T=300))



defaultdict(<class 'int'>, {'C[d](C[d])(F)2': 2, 'F(C[d])': 4})
-702.4097261378926
65.43301534389283


## 2. Batch Comparison Against DFT Data

Loop over all molecules in the DFT thermochemistry dataset, estimate enthalpy at 300 K
with pGrAdd, and compute the absolute error relative to the DFT reference value.

Molecules for which pGrAdd cannot assign all groups are flagged as `FAIL` and assigned
`NaN` so they do not contaminate the error statistics.

**Input file:** `Data/fluorocarbon_DFT_thermochemistry.csv`  
**Output file:** `Data/DFT_with_pGrAdd_Enthalpy_and_Error.csv`

In [ ]:
import pandas as pd
import numpy as np
from pgradd.GroupAdd.Library import GroupLibrary
import pgradd.ThermoChem

# ---------------------------------------------------------
# Load Benson GAF library
# ---------------------------------------------------------
lib = GroupLibrary.Load('BensonGAF')

# ---------------------------------------------------------
# Load your DFT dataset
# ---------------------------------------------------------
df = pd.read_csv("Data/fluorocarbon_DFT_thermochemistry.csv")

# Ensure required columns exist
required_cols = ["SMILES", "Enthalpy"]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

# ---------------------------------------------------------
# Storage lists
# ---------------------------------------------------------
pgradd_H = []
errors = []
status = []   # success/fail flag
messages = [] # error messages

# ---------------------------------------------------------
# Loop over molecules
# ---------------------------------------------------------
for smi in df["SMILES"]:
    try:
        # Get descriptors
        descriptors = lib.GetDescriptors(smi)

        # Estimate thermochemistry
        thermochem = lib.Estimate(descriptors, "thermochem")

        # Compute enthalpy at 300 K (kcal/mol)
        H_300 = thermochem.get_H(units="kcal/mol", T=300)

        pgradd_H.append(H_300)
        status.append("OK")
        messages.append("")

    except Exception as e:
        # If pGrAdd fails, store NaN and record the error
        pgradd_H.append(np.nan)
        status.append("FAIL")
        messages.append(str(e))

# ---------------------------------------------------------
# Add results to dataframe
# ---------------------------------------------------------
df["pGrAdd_H_300K"] = pgradd_H

# Absolute error vs DFT enthalpy
df["Abs_Error"] = np.abs(df["pGrAdd_H_300K"] - df["Enthalpy"])

# Status + error messages for debugging
df["Status"] = status
df["Message"] = messages

# ---------------------------------------------------------
# Save output
# ---------------------------------------------------------
df.to_csv("Data/DFT_with_pGrAdd_Enthalpy_and_Error.csv", index=False)

print("Finished computing pGrAdd enthalpies.")
print(df[["SMILES", "Enthalpy", "pGrAdd_H_300K", "Abs_Error", "Status"]].head())